# Élagage des variables, à partir du scénario complet `ml_11`

**But** : retirer les groupes de colonnes qui n'apportent rien, pour obtenir une version allégée (moins de bruit, modèles plus lisibles), qui deviendra le scénario `ml_12`, lancé normalement avec la boucle et comparé à `ml_4`.

**Protocole** (fixé avant de lancer, voir `methodologie_ML.md`, section 9) :
1. **RandomForest débridé** : le diagnostic de capacité a montré que la règle de surapprentissage le bride un peu. Sa profondeur est choisie ici **de 1 en 1**, de 8 à 16 (feuilles de 20 clients, 500 arbres), sur toutes les colonnes. Ce réglage est choisi sur les mêmes plis : son score est un peu optimiste, il sert à comparer des versions entre elles, pas à être cité comme résultat.
2. **Réglages figés** pour les autres modèles, ceux retenus dans `ml_11` : régression logistique, CatBoost, MLP. Le SVM (trop lent) et le KNN (dilué par les colonnes) sont laissés de côté ; ils seront relancés dans `ml_12`.
3. **Importance par permutation de chaque groupe**, sur les plis de validation : elle fixe l'**ordre** des retraits (du moins important au plus important), pas la décision.
4. **Une seule passe de retraits**, groupe par groupe, dans cet ordre : le groupe est retiré si le score décisionnel F2 **meilleur contre meilleur**, pli par pli, ne baisse pas au-delà de l'écart-type des gains (verdict « perte » = le groupe reste). La comparaison se fait toujours avec la **version complète** (toutes les colonnes), jamais avec la version précédente : sinon, de petites pertes « dans le bruit » à chaque retrait s'additionnent.

Rien n'est écrit dans le fichier de résultats, aucun modèle n'est enregistré, le test n'est pas lu.

## Données, variables et pipeline : identiques à `ml_11` (toutes les colonnes)

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

# Racine du projet : premier dossier parent qui contient 'data'
current_dir = Path.cwd()
root_dir = next(p for p in [current_dir] + list(current_dir.parents) if (p / "data").exists())

train = pd.read_csv(root_dir / "data" / "ML" / "train_dataset.csv")
print(f"Clients du train : {len(train)} ; colonnes disponibles : {train.shape[1]}")

Clients du train : 19358 ; colonnes disponibles : 56


In [2]:
# ==============================================================================
# VARIABLES DU SCÉNARIO : seule cellule à modifier pour changer les colonnes
# ==============================================================================

# Colonnes ajoutées par ce scénario, rangées selon leur encodage
ajout_num = ["ratio_PAY_BILL_global", "ratio_PAY_BILL_median", "ratio_PAY_BILL_regularite", "ratio_PAY_BILL1", "ratio_PAY_BILL2", "ratio_PAY_BILL3", "ratio_PAY_BILL4", "ratio_PAY_BILL5", "ratio_BILL_LIMIT1", "ratio_BILL_LIMIT2", "ratio_BILL_LIMIT3", "ratio_BILL_LIMIT4", "ratio_BILL_LIMIT5", "ratio_BILL_LIMIT6", "NB_MOIS_CTX", "CUMUL_INCIDENT"]   # nombres, montants, ratios, compteurs
ajout_log = ["BILL_evol_relatif", "PAYAMT_evol_relatif"]   # montants positifs ou nuls très étirés : logarithme, puis mise à l'échelle
ajout_log_signe = ["BILL_evol_brut", "PAYAMT_evol_brut"]   # montants qui peuvent être négatifs (avoirs) : logarithme signé, puis mise à l'échelle
ajout_cat = ["FLAG_OUVERTURE", "FLAG_DEGEL", "MOIS_ACTIVATION", "TYPE_USAGE", "PAY_habituel", "FAUX_CODAGE", "SURVEILLANCE_RECENTE", "FLAG_CTX", "MOIS_SORTIE_CTX", "FLAG_RETARD", "MOIS_SORTIE_RETARD"]   # catégories sans ordre (une colonne par catégorie), indicateurs 0 / 1
ajout_ord = []   # catégories dans un ordre naturel (encodage ordinal)

# Colonnes du socle retirées par ce scénario
retirees = []

# ------------------------------------------------------------------------------
# Socle : les 23 variables d'origine (ne pas modifier)
# PAY_n : des nombres déjà ordonnés et régulièrement espacés (-2 à 8, un cran par mois de retard), mis à l'échelle comme les autres (v6)
# PAY_AMTn : paiements très étirés (médiane d'environ 2 000 NT$, maximum de plus de 400 000 NT$), passés au logarithme (v7)
# BILL_AMTn : factures, traitées comme les paiements qu'elles appellent, avec un logarithme signé (avoirs négatifs) (v8)
socle_num = ["LIMIT_BAL", "AGE"] + [f"PAY_{n}" for n in range(1, 7)]
socle_log = [f"PAY_AMT{n}" for n in range(1, 7)]
socle_log_signe = [f"BILL_AMT{n}" for n in range(1, 7)]
socle_cat = ["SEX", "EDUCATION", "MARRIAGE"]
socle_ord = []

# ------------------------------------------------------------------------------
# Remplacements décidés dans docs/colonnes_creees.md (colonne « Adaptation et usage ML ») : {colonne: (valeurs remplacées, valeur)}
# Faits dans le pipeline, avant la transformation propre au type de la colonne (mise à l'échelle, logarithme, catégories…),
# et donc enregistrés avec le modèle. Valeur précise décidée par le métier, jamais calculée sur les clients (ni médiane ni moyenne).
# Actifs seulement si la colonne est une variable du scénario, quelle que soit sa liste. Un vide s'écrit np.nan ; une catégorie
# s'écrit en texte. À compléter à chaque nouvelle décision
remplacements = {
    "ratio_PAY_BILL_global": ([np.nan], 100),          # dû nul ou négatif : rien à payer, client à jour
    "ratio_PAY_BILL_median": ([-1], 100),              # seule la facture de septembre est due
    "ratio_PAY_BILL_regularite": ([np.nan, -1], 100),  # une seule facture due, ou « Première facture »
}

num_col = [c for c in socle_num if c not in retirees] + ajout_num
log_col = [c for c in socle_log if c not in retirees] + ajout_log
log_signe_col = [c for c in socle_log_signe if c not in retirees] + ajout_log_signe
cat_col = [c for c in socle_cat if c not in retirees] + ajout_cat
ord_col = [c for c in socle_ord if c not in retirees] + ajout_ord
variables = num_col + log_col + log_signe_col + cat_col + ord_col

# Contrôles : colonnes présentes dans le train, aucune en double, ni identifiant ni cible
absentes = [c for c in variables + retirees if c not in train.columns]
doublons = sorted({c for c in variables if variables.count(c) > 1})
interdites = [c for c in variables if c in ("ID", "dpnm")]
assert not absentes, f"Colonnes absentes du train : {absentes} (à créer dans creation_datasets_ML.ipynb)"
assert not doublons, f"Colonnes en double : {doublons}"
assert not interdites, f"Colonnes interdites comme variables : {interdites}"
# Logarithme : montants positifs ou nuls, une fois les valeurs remplacées mises de côté
for c in log_col:
    valeurs = train[c].dropna()
    if c in remplacements:
        valeurs = valeurs[~valeurs.isin([v for v in remplacements[c][0] if not pd.isna(v)])]
    assert (valeurs >= 0).all(), f"Le logarithme ne s'applique qu'à des montants positifs ou nuls : {c}"
# Remplacements : aucune variable ne garde un vide sans remplacement décidé
remplacees = [c for c in variables if c in remplacements]
vides_sans_remplacement = [c for c in variables if train[c].isna().any()
                           and not (c in remplacements and any(pd.isna(v) for v in remplacements[c][0]))]
assert not vides_sans_remplacement, f"Colonnes avec des vides, sans remplacement décidé dans docs/colonnes_creees.md : {vides_sans_remplacement}"

X_train = train[variables].reset_index(drop=True)
y_train = train["dpnm"].reset_index(drop=True)

print(f"Variables        : {len(variables)} ({len(num_col)} numériques, {len(log_col)} au logarithme, {len(log_signe_col)} au logarithme signé, {len(cat_col)} catégorielles, {len(ord_col)} ordinales)")
print(f"Ajoutées         : {ajout_num + ajout_log + ajout_log_signe + ajout_cat + ajout_ord if ajout_num + ajout_log + ajout_log_signe + ajout_cat + ajout_ord else 'aucune'}")
print(f"Retirées         : {retirees if retirees else 'aucune'}")
print(f"Remplacements    : {', '.join(f'{c} ({remplacements[c][0]} -> {remplacements[c][1]})' for c in remplacees) if remplacees else 'aucun'}")
print(f"Défauts          : {y_train.sum()} ({y_train.mean() * 100:.2f} %)")

Variables        : 54 (24 numériques, 8 au logarithme, 8 au logarithme signé, 14 catégorielles, 0 ordinales)
Ajoutées         : ['ratio_PAY_BILL_global', 'ratio_PAY_BILL_median', 'ratio_PAY_BILL_regularite', 'ratio_PAY_BILL1', 'ratio_PAY_BILL2', 'ratio_PAY_BILL3', 'ratio_PAY_BILL4', 'ratio_PAY_BILL5', 'ratio_BILL_LIMIT1', 'ratio_BILL_LIMIT2', 'ratio_BILL_LIMIT3', 'ratio_BILL_LIMIT4', 'ratio_BILL_LIMIT5', 'ratio_BILL_LIMIT6', 'NB_MOIS_CTX', 'CUMUL_INCIDENT', 'BILL_evol_relatif', 'PAYAMT_evol_relatif', 'BILL_evol_brut', 'PAYAMT_evol_brut', 'FLAG_OUVERTURE', 'FLAG_DEGEL', 'MOIS_ACTIVATION', 'TYPE_USAGE', 'PAY_habituel', 'FAUX_CODAGE', 'SURVEILLANCE_RECENTE', 'FLAG_CTX', 'MOIS_SORTIE_CTX', 'FLAG_RETARD', 'MOIS_SORTIE_RETARD']
Retirées         : aucune
Remplacements    : ratio_PAY_BILL_global ([nan] -> 100), ratio_PAY_BILL_median ([-1] -> 100), ratio_PAY_BILL_regularite ([nan, -1] -> 100)
Défauts          : 3084 (15.93 %)


In [3]:
from catboost import CatBoostClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
import numpy as np
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.svm import SVC

# Pondération des défauts pour CatBoost (part des non-défauts / part des défauts)
scale_pos = (y_train == 0).sum() / (y_train == 1).sum()

# Transformation de chaque type de colonne (un nouvel objet à chaque appel)
TRANSFORMATIONS = {
    "num": lambda: StandardScaler(),
    "log": lambda: make_pipeline(FunctionTransformer(np.log1p, feature_names_out="one-to-one"), StandardScaler()),
    "log_signe": lambda: make_pipeline(FunctionTransformer(np.arcsinh, feature_names_out="one-to-one"), StandardScaler()),
    "nom": lambda: OneHotEncoder(sparse_output=False, handle_unknown="ignore"),
    "ord": lambda: make_pipeline(OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), StandardScaler()),
}
COLONNES_PAR_TYPE = {"num": num_col, "log": log_col, "log_signe": log_signe_col, "nom": cat_col, "ord": ord_col}

def avec_remplacement(colonne, transformation):
    # Valeurs remplacées par la valeur décidée (cellule « variables du scénario »), puis transformation propre au type
    valeurs, valeur = remplacements[colonne]
    # Le vide d'abord : SimpleImputer refuse une colonne qui contient des vides quand il cherche une autre valeur
    valeurs = sorted(valeurs, key=lambda v: not pd.isna(v))
    return make_pipeline(*[SimpleImputer(missing_values=v, strategy="constant", fill_value=valeur) for v in valeurs], transformation)

# Une branche par type pour les colonnes sans remplacement, une branche par colonne avec remplacement.
# Sans remplacement (le socle), le pipeline est exactement celui de la v8
preprocessor = ColumnTransformer(transformers=
    [(nom, TRANSFORMATIONS[nom](), [c for c in colonnes if c not in remplacements]) for nom, colonnes in COLONNES_PAR_TYPE.items()]
    + [(f"remplacement_{c}", avec_remplacement(c, TRANSFORMATIONS[nom]()), [c])
       for nom, colonnes in COLONNES_PAR_TYPE.items() for c in colonnes if c in remplacements])

# Objectif de détection : part des défauts que le modèle doit trouver (60 % pour commencer, objectif 70 %)
RAPPEL_MINIMAL = 0.60

def seuil_pour_rappel(y, proba, rappel=RAPPEL_MINIMAL):
    # Seuil le plus haut qui détecte au moins la part voulue des défauts
    proba_defauts = np.sort(proba[np.asarray(y) == 1])[::-1]
    return proba_defauts[int(np.ceil(rappel * len(proba_defauts))) - 1]

def score_decision(estimator, X, y):
    # Précision au seuil qui détecte au moins RAPPEL_MINIMAL des défauts (score optimisé par la boucle)
    proba = estimator.predict_proba(X)[:, 1]
    signales = proba >= seuil_pour_rappel(y, proba)
    return np.asarray(y)[signales].mean()

scorers = {
    "decision": score_decision,
    "roc_auc": "roc_auc",
    "average_precision": "average_precision",
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Modèles lancés dans ce notebook : retirer un nom pour ne pas le lancer (le reste du notebook s'adapte)
MODELES_A_TESTER = ["LogisticRegression", "SVM", "MLPClassifier", "KNN", "RandomForest", "CatBoost"]

def modeles_de_base():
    # Les 6 modèles d'origine, avec leurs réglages de la première itération ; seuls ceux de MODELES_A_TESTER sont rendus
    tous = {
        "LogisticRegression": make_pipeline(preprocessor, LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
        "SVM": make_pipeline(preprocessor, CalibratedClassifierCV(estimator=SVC(class_weight="balanced", random_state=42), ensemble=False)),
        "MLPClassifier": make_pipeline(preprocessor, MLPClassifier(hidden_layer_sizes=(50, 25), max_iter=500, random_state=42, early_stopping=True)),
        "KNN": make_pipeline(preprocessor, KNeighborsClassifier(n_neighbors=5, n_jobs=-1)),
        "RandomForest": make_pipeline(preprocessor, RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=-1)),
        "CatBoost": make_pipeline(preprocessor, CatBoostClassifier(scale_pos_weight=scale_pos, random_state=42, verbose=0, thread_count=-1, allow_writing_files=False)),
    }
    return {nom: tous[nom] for nom in MODELES_A_TESTER}

In [4]:
from IPython.display import Markdown, display
from sklearn.metrics import fbeta_score
import time

y = y_train.to_numpy()
plis = list(cv.split(X_train, y_train))   # les mêmes 5 plis que tous les scénarios

def f2_au_rappel_minimal(y_vrai, proba):
    return fbeta_score(y_vrai, proba >= seuil_pour_rappel(y_vrai, proba), beta=2)

def preprocesseur(gardees):
    # Même pipeline que les scénarios, limité aux colonnes gardées
    return ColumnTransformer(transformers=
        [(nom, TRANSFORMATIONS[nom](), [c for c in colonnes if c in gardees and c not in remplacements]) for nom, colonnes in COLONNES_PAR_TYPE.items()]
        + [(f"remplacement_{c}", avec_remplacement(c, TRANSFORMATIONS[nom]()), [c])
           for nom, colonnes in COLONNES_PAR_TYPE.items() for c in colonnes if c in gardees and c in remplacements])

def evaluer(gardees, fabriques, garder_modeles=False):
    """Score décisionnel F2 de chaque pli, pour chaque modèle, avec les seules colonnes gardées."""
    f2, ajustes = {}, {}
    for nom, fabrique in fabriques.items():
        f2[nom], ajustes[nom] = [], []
        for entrainement, validation in plis:
            pipeline = make_pipeline(preprocesseur(gardees), fabrique())
            pipeline.fit(X_train.iloc[entrainement][gardees], y[entrainement])
            f2[nom].append(f2_au_rappel_minimal(y[validation], pipeline.predict_proba(X_train.iloc[validation][gardees])[:, 1]))
            if garder_modeles:
                ajustes[nom].append(pipeline)
    return (f2, ajustes) if garder_modeles else f2

def meilleur(f2):
    return max(f2, key=lambda nom: np.mean(f2[nom]))

def verdict(gains):
    moyenne, ecart_type = np.mean(gains), np.std(gains)
    return "✅ gain réel" if moyenne > ecart_type else ("❌ perte" if moyenne < -ecart_type else "➖ dans le bruit")

## 1. Profondeur de RandomForest, de 1 en 1

Feuilles de 20 clients et 500 arbres (le meilleur réglage du diagnostic de capacité), profondeur de 8 à 16, toutes les colonnes. La profondeur retenue est celle du meilleur score décisionnel F2 moyen.

In [5]:
PROFONDEURS_RF = list(range(8, 17))
resultats_rf = []
for profondeur in PROFONDEURS_RF:
    debut = time.time()
    f2 = evaluer(variables, {"RandomForest": lambda p=profondeur: RandomForestClassifier(
        n_estimators=500, max_depth=p, min_samples_leaf=20, max_features="sqrt", class_weight="balanced", random_state=42, n_jobs=-1)})
    resultats_rf.append({"Profondeur": profondeur, "Score décisionnel F2": np.mean(f2["RandomForest"]),
                         "Écart-type": np.std(f2["RandomForest"]), "Durée (min)": (time.time() - debut) / 60})
df_rf = pd.DataFrame(resultats_rf).set_index("Profondeur")
display(df_rf.round(4))
PROFONDEUR_RF = int(df_rf["Score décisionnel F2"].idxmax())
print(f"Profondeur retenue pour RandomForest : {PROFONDEUR_RF}")

,Score décisionnel F2,Écart-type,Durée (min)
Profondeur,,,
8,0.4846,0.0044,0.0657
9,0.4852,0.0059,0.0660
10,0.4859,0.0058,0.0668
11,0.4862,0.0054,0.0654
12,0.4890,0.0050,0.0660
13,0.4889,0.0048,0.0680
14,0.4869,0.0056,0.0691
15,0.4878,0.0056,0.0671
16,0.4887,0.0054,0.0692


Profondeur retenue pour RandomForest : 12


**Comment lire** : une ligne par profondeur. Les écarts entre profondeurs voisines sont souvent plus petits que l'écart-type : la profondeur retenue est la meilleure de ces essais, sans que les voisines soient nettement moins bonnes.

## 2. Modèles figés et importance de chaque groupe

Les quatre modèles sont entraînés une fois sur toutes les colonnes. Pour chaque groupe, on mélange ses colonnes (le même mélange pour toutes) dans les clients de validation, et on mesure la baisse du score décisionnel F2, moyennée sur les plis et les modèles. Un groupe dont le mélange ne change rien est peu utilisé, ou redondant avec un autre.

In [6]:
MODELES_FIGES = {
    "LogisticRegression": lambda: LogisticRegression(C=0.01, max_iter=1000, class_weight="balanced", random_state=42),
    "CatBoost": lambda: CatBoostClassifier(depth=2, iterations=445, l2_leaf_reg=10.0, learning_rate=0.00887, scale_pos_weight=scale_pos,
                                           random_state=42, verbose=0, thread_count=-1, allow_writing_files=False),
    "RandomForest": lambda: RandomForestClassifier(n_estimators=500, max_depth=PROFONDEUR_RF, min_samples_leaf=20, max_features="sqrt",
                                                   class_weight="balanced", random_state=42, n_jobs=-1),
    "MLPClassifier": lambda: MLPClassifier(hidden_layer_sizes=(32, 16), alpha=0.1, max_iter=500, random_state=42, early_stopping=True),
}

GROUPES = {
    "Ratios mensuels": [f"ratio_PAY_BILL{n}" for n in range(1, 6)],
    "Utilisation du plafond": [f"ratio_BILL_LIMIT{n}" for n in range(1, 7)],
    "Comportement de paiement": ["ratio_PAY_BILL_median", "ratio_PAY_BILL_regularite", "TYPE_USAGE"],
    "ml_4 (taux global et vie du compte)": ["ratio_PAY_BILL_global", "FLAG_OUVERTURE", "FLAG_DEGEL", "MOIS_ACTIVATION"],
    "Rupture de comportement": ["BILL_evol_brut", "PAYAMT_evol_brut", "BILL_evol_relatif", "PAYAMT_evol_relatif"],
    "Historique des retards": ["FLAG_CTX", "MOIS_SORTIE_CTX", "NB_MOIS_CTX", "FLAG_RETARD", "MOIS_SORTIE_RETARD"],
    "Résumé des codifications": ["PAY_habituel", "CUMUL_INCIDENT"],
    "Traces des corrections": ["FAUX_CODAGE", "SURVEILLANCE_RECENTE"],
    "Démographie (socle)": ["SEX", "EDUCATION", "MARRIAGE", "AGE"],
}
absentes = [c for g in GROUPES.values() for c in g if c not in variables]
assert not absentes, f"Colonnes des groupes absentes des variables : {absentes}"

debut = time.time()
f2_complet, ajustes = evaluer(variables, MODELES_FIGES, garder_modeles=True)
print(f"Toutes les colonnes ({len(variables)}) : " + " ; ".join(f"{nom} {np.mean(v):.3f}" for nom, v in f2_complet.items())
      + f" ; meilleur : {meilleur(f2_complet)} ({(time.time() - debut) / 60:.1f} min)")

generateur = np.random.default_rng(42)
importance = {}
for groupe, colonnes in GROUPES.items():
    baisses = []
    for nom in MODELES_FIGES:
        for k, (_, validation) in enumerate(plis):
            X_val = X_train.iloc[validation][variables].copy()
            ordre = generateur.permutation(len(X_val))
            X_val[colonnes] = X_val[colonnes].to_numpy()[ordre]          # mêmes clients mélangés pour toutes les colonnes du groupe
            baisses.append(f2_complet[nom][k] - f2_au_rappel_minimal(y[validation], ajustes[nom][k].predict_proba(X_val)[:, 1]))
    importance[groupe] = np.mean(baisses)
df_importance = pd.Series(importance, name="Baisse du score décisionnel F2 quand le groupe est mélangé").sort_values()
display(df_importance.round(4).to_frame())
ORDRE_RETRAITS = list(df_importance.index)

Toutes les colonnes (54) : LogisticRegression 0.489 ; CatBoost 0.486 ; RandomForest 0.489 ; MLPClassifier 0.479 ; meilleur : RandomForest (0.2 min)


,Baisse du score décisionnel F2 quand le groupe est mélangé
Traces des corrections,0.0006
ml_4 (taux global et vie du compte),0.0011
Comportement de paiement,0.0021
Ratios mensuels,0.0033
Rupture de comportement,0.0034
Démographie (socle),0.0037
Historique des retards,0.0051
Utilisation du plafond,0.0098
Résumé des codifications,0.0133


**Comment lire** : plus la baisse est grande, plus les modèles s'appuient sur le groupe. Une baisse nulle ou négative : le groupe ne sert pas, ou un autre groupe porte la même information. Les retraits sont essayés dans cet ordre, du haut vers le bas.

## 3. Retraits, groupe par groupe

Pour chaque groupe, dans l'ordre ci-dessus : les quatre modèles sont réentraînés sans lui (et sans les groupes déjà retirés), puis le meilleur modèle est comparé, pli par pli, au meilleur modèle de la **version complète** (toutes les colonnes), jamais à la version précédente, pour que de petites pertes ne s'additionnent pas. Retrait **gardé** si le verdict n'est pas une perte ; sinon le groupe **reste**.

In [7]:
gardees = list(variables)
f2_courant = f2_complet
journal = []
for groupe in ORDRE_RETRAITS:
    debut = time.time()
    candidat = [c for c in gardees if c not in GROUPES[groupe]]
    f2_candidat = evaluer(candidat, MODELES_FIGES)
    avant, apres = meilleur(f2_complet), meilleur(f2_candidat)      # comparaison avec la version complète
    gains = np.array(f2_candidat[apres]) - np.array(f2_complet[avant])
    v = verdict(gains)
    retire = v != "❌ perte"
    journal.append({"Groupe": groupe, "Meilleur, toutes les colonnes": f"{avant} {np.mean(f2_complet[avant]):.3f}",
                    "Meilleur sans le groupe": f"{apres} {np.mean(f2_candidat[apres]):.3f}",
                    "Gain": f"{gains.mean():+.3f} ± {gains.std():.3f} ({(gains > 0).sum()} plis sur 5)",
                    "Verdict": v, "Décision": "retiré" if retire else "gardé"})
    if retire:
        gardees, f2_courant = candidat, f2_candidat
    print(f"{groupe} : {v}, {'retiré' if retire else 'gardé'} ({(time.time() - debut) / 60:.1f} min)")
display(pd.DataFrame(journal).set_index("Groupe"))

Traces des corrections : ➖ dans le bruit, retiré (0.4 min)
ml_4 (taux global et vie du compte) : ➖ dans le bruit, retiré (0.4 min)
Comportement de paiement : ➖ dans le bruit, retiré (0.3 min)
Ratios mensuels : ➖ dans le bruit, retiré (0.3 min)
Rupture de comportement : ➖ dans le bruit, retiré (0.3 min)
Démographie (socle) : ➖ dans le bruit, retiré (0.3 min)
Historique des retards : ➖ dans le bruit, retiré (0.3 min)
Utilisation du plafond : ➖ dans le bruit, retiré (0.3 min)
Résumé des codifications : ❌ perte, gardé (0.3 min)


,"Meilleur, toutes les colonnes",Meilleur sans le groupe,Gain,Verdict,Décision
Groupe,,,,,
Traces des corrections,RandomForest 0.489,LogisticRegression 0.489,-0.000 ± 0.004 (1 plis sur 5),➖ dans le bruit,retiré
ml_4 (taux global et vie du compte),RandomForest 0.489,RandomForest 0.489,-0.000 ± 0.002 (3 plis sur 5),➖ dans le bruit,retiré
Comportement de paiement,RandomForest 0.489,RandomForest 0.488,-0.001 ± 0.003 (2 plis sur 5),➖ dans le bruit,retiré
Ratios mensuels,RandomForest 0.489,LogisticRegression 0.487,-0.002 ± 0.004 (2 plis sur 5),➖ dans le bruit,retiré
Rupture de comportement,RandomForest 0.489,LogisticRegression 0.488,-0.001 ± 0.004 (2 plis sur 5),➖ dans le bruit,retiré
Démographie (socle),RandomForest 0.489,LogisticRegression 0.486,-0.003 ± 0.008 (1 plis sur 5),➖ dans le bruit,retiré
Historique des retards,RandomForest 0.489,CatBoost 0.486,-0.003 ± 0.006 (3 plis sur 5),➖ dans le bruit,retiré
Utilisation du plafond,RandomForest 0.489,RandomForest 0.485,-0.004 ± 0.005 (1 plis sur 5),➖ dans le bruit,retiré
Résumé des codifications,RandomForest 0.489,RandomForest 0.483,-0.006 ± 0.006 (0 plis sur 5),❌ perte,gardé


**Comment lire** : une ligne par groupe, dans l'ordre des essais. « Gain » compare le meilleur modèle sans le groupe (et sans ceux déjà retirés) au meilleur modèle avec toutes les colonnes : un gain positif veut dire que le retrait améliore le score (le groupe ajoutait du bruit).

## 4. Version élaguée, à reprendre dans `ml_12`

Les listes ci-dessous sont à recopier telles quelles dans la cellule « variables du scénario » de `ml_12` (copie de `ml_0`, référence `"4"`).

In [8]:
retires = [g for g, colonnes in GROUPES.items() if not any(c in gardees for c in colonnes)]
display(Markdown(f"**Groupes retirés** : {', '.join(retires) if retires else 'aucun'} ; **variables gardées** : {len(gardees)} sur {len(variables)} ; "
                 f"meilleur modèle : {meilleur(f2_courant)} (score décisionnel F2 {np.mean(f2_courant[meilleur(f2_courant)]):.3f}, "
                 f"contre {np.mean(f2_complet[meilleur(f2_complet)]):.3f} avec toutes les colonnes)"))
socle = socle_num + socle_log + socle_log_signe + socle_cat + socle_ord
print("ajout_num =", [c for c in ajout_num if c in gardees])
print("ajout_log =", [c for c in ajout_log if c in gardees])
print("ajout_log_signe =", [c for c in ajout_log_signe if c in gardees])
print("ajout_cat =", [c for c in ajout_cat if c in gardees])
print("ajout_ord =", [c for c in ajout_ord if c in gardees])
print("retirees =", [c for c in socle if c not in gardees])

**Groupes retirés** : Ratios mensuels, Utilisation du plafond, Comportement de paiement, ml_4 (taux global et vie du compte), Rupture de comportement, Historique des retards, Traces des corrections, Démographie (socle) ; **variables gardées** : 21 sur 54 ; meilleur modèle : RandomForest (score décisionnel F2 0.485, contre 0.489 avec toutes les colonnes)

ajout_num = ['CUMUL_INCIDENT']
ajout_log = []
ajout_log_signe = []
ajout_cat = ['PAY_habituel']
ajout_ord = []
retirees = ['AGE', 'SEX', 'EDUCATION', 'MARRIAGE']
